In [2]:
from ten_vad import TenVad
import numpy as np
import scipy.io.wavfile as wav

# загрузка файла
sr, audio = wav.read("../../data/audio/audio-test.wav")  # audio в int16, 16 kHz
if audio.ndim == 2:
        audio = audio.mean(axis=1)
        audio = (audio * 32768).astype(np.int16)
vad = TenVad(hop_size=256, threshold=0.5)  # hop_size = размер фрейма
speech_segments = []

# разбиваем на фреймы
for i in range(0, len(audio), vad.hop_size):
    frame = audio[i : i + vad.hop_size]
    if len(frame) < vad.hop_size:
        break
    prob, flag = vad.process(frame)  # flag = 1 если речь
    speech_segments.append((prob, flag))

In [3]:
from ten_vad import TenVad
import numpy as np
import scipy.io.wavfile as wav

# -----------------------------
# 1) Load audio (int16, 16 kHz)
# -----------------------------
sr, audio = wav.read("../../data/audio/audio-test.wav")

if audio.ndim == 2:
    # safe mono: average in int32 to avoid overflow, then cast back
    audio = audio.astype(np.int32).mean(axis=1)
    audio = np.clip(np.rint(audio), -32768, 32767).astype(np.int16)

# -----------------------------
# 2) TEN-VAD frame inference
# -----------------------------
vad = TenVad(hop_size=256, threshold=0.5)

frames = []  # store (prob, flag)
num_frames = len(audio) // vad.hop_size

for k in range(num_frames):
    i = k * vad.hop_size
    frame = audio[i : i + vad.hop_size]
    prob, flag = vad.process(frame)  # returns (probability, flag) :contentReference[oaicite:1]{index=1}
    frames.append((prob, int(flag)))

# -----------------------------
# 3) Convert frame flags -> speech intervals
# -----------------------------
speech_segments = []  # list of dicts with start/end (+ optional stats)

in_speech = False
start_frame = None
prob_sum = 0.0
n_prob = 0

for idx, (prob, flag) in enumerate(frames):
    if flag == 1 and not in_speech:
        in_speech = True
        start_frame = idx
        prob_sum = float(prob)
        n_prob = 1

    elif flag == 1 and in_speech:
        prob_sum += float(prob)
        n_prob += 1

    elif flag == 0 and in_speech:
        end_frame = idx  # exclusive
        t_start = (start_frame * vad.hop_size) / sr
        t_end = (end_frame * vad.hop_size) / sr

        speech_segments.append({
            "start": t_start,
            "end": t_end,
            "flag": 1,
            "mean_prob": prob_sum / max(n_prob, 1),
            "n_frames": n_prob,
        })

        in_speech = False
        start_frame = None

# close tail segment if file ended during speech
if in_speech and start_frame is not None:
    end_frame = len(frames)
    t_start = (start_frame * vad.hop_size) / sr
    t_end = (end_frame * vad.hop_size) / sr

    speech_segments.append({
        "start": t_start,
        "end": t_end,
        "flag": 1,
        "mean_prob": prob_sum / max(n_prob, 1),
        "n_frames": n_prob,
    })

print("num speech segments:", len(speech_segments))
print("first 5:", speech_segments[:5])

num speech segments: 733
first 5: [{'start': 0.09287981859410431, 'end': 0.11029478458049886, 'flag': 1, 'mean_prob': 0.5236969788869222, 'n_frames': 3}, {'start': 0.24961451247165534, 'end': 0.2612244897959184, 'flag': 1, 'mean_prob': 0.5548204481601715, 'n_frames': 2}, {'start': 0.26702947845804986, 'end': 0.2786394557823129, 'flag': 1, 'mean_prob': 0.5277848541736603, 'n_frames': 2}, {'start': 0.28444444444444444, 'end': 0.35410430839002266, 'flag': 1, 'mean_prob': 0.6020006636778513, 'n_frames': 12}, {'start': 0.3947392290249433, 'end': 0.7952834467120181, 'flag': 1, 'mean_prob': 0.7136497540750365, 'n_frames': 69}]


In [4]:
from typing import List, Dict

def merge_gap_dict(segments: List[Dict], gap: float) -> List[Dict]:
    """
    Merge adjacent speech segments if gap between them < gap seconds.
    Keeps stats: n_frames sum; mean_prob as weighted mean by n_frames.
    Assumes segments are sorted by start time and all are speech (flag=1).
    """
    if not segments:
        return []

    merged = []
    cur = dict(segments[0])  # copy

    for seg in segments[1:]:
        # gap between current end and next start
        if seg["start"] <= cur["end"] + gap:
            # merge
            old_frames = cur.get("n_frames", 0)
            new_frames = seg.get("n_frames", 0)

            # update time bounds
            cur["end"] = max(cur["end"], seg["end"])

            # update n_frames
            cur["n_frames"] = old_frames + new_frames

            # weighted mean probability (if present)
            if "mean_prob" in cur and "mean_prob" in seg and (old_frames + new_frames) > 0:
                cur["mean_prob"] = (
                    cur["mean_prob"] * old_frames + seg["mean_prob"] * new_frames
                ) / (old_frames + new_frames)

            cur["flag"] = 1
        else:
            merged.append(cur)
            cur = dict(seg)

    merged.append(cur)
    return merged

In [5]:
merge_spech_seg = merge_gap_dict(speech_segments, 0.3)

In [9]:
from typing import List, Dict, Tuple

def filter_min_speech_dict(segments: List[Dict], min_speech: float) -> List[Dict]:
    """
    Keep only speech segments with duration >= min_speech seconds.
    segments: list of dicts with "start", "end"
    """
    out = []
    for seg in segments:
        if (seg["end"] - seg["start"]) >= min_speech:
            out.append(seg)
    return out

In [7]:
speech_seg_merg_min = filter_min_speech_dict(merge_spech_seg, 0.5)

<h1>Sliding Windows

In [10]:
def make_sliding_windows_from_speech(
    speech_segments: List[Dict],
    window_sec: float = 2.0,
    step_sec: float = 0.5,
) -> List[Tuple[float, float]]:
    """
    speech_segments: list of dicts with keys {"start","end"} in seconds (original timeline)
    returns: list of (t0, t1) windows in seconds (original timeline)
    """
    windows: List[Tuple[float, float]] = []

    for seg in speech_segments:
        S = float(seg["start"])
        E = float(seg["end"])
        t = S

        # only windows fully inside [S, E]
        while (t + window_sec) <= E:
            windows.append((t, t + window_sec))
            t += step_sec

    return windows

def slice_waveform_int16(
    audio_int16: np.ndarray,
    sr: int,
    t0: float,
    t1: float,
) -> np.ndarray:
    """
    Returns int16 audio chunk corresponding to [t0, t1) seconds.
    """
    i0 = int(round(t0 * sr))
    i1 = int(round(t1 * sr))
    i0 = max(0, min(i0, len(audio_int16)))
    i1 = max(0, min(i1, len(audio_int16)))
    return audio_int16[i0:i1]


def int16_to_float32(audio_int16: np.ndarray) -> np.ndarray:
    """
    Convert int16 PCM to float32 in [-1, 1).
    """
    return audio_int16.astype(np.float32) / 32768.0

In [ ]:
window_sec = 2.0
step_sec = 0.5

windows = make_sliding_windows_from_speech(
    speech_seg_merg_min,
    window_sec=window_sec,
    step_sec=step_sec,
)

print("num speech segments:", len(speech_seg_merg_min))
print("num windows:", len(windows))
print("first 5 windows:", windows[:5])

# Build chunks for embedding inference
chunks_for_embedding = []
for (t0, t1) in windows:
    chunk_int16 = slice_waveform_int16(audio, sr, t0, t1)
    chunk =       # float32 [-1,1)
    chunks_for_embedding.append((t0, t1, chunk))

print("first chunk:", chunks_for_embedding[0][0], chunks_for_embedding[0][1], chunks_for_embedding)

num speech segments: 47
num windows: 122
first 5 windows: [(0.09287981859410431, 2.0928798185941044), (0.5928798185941043, 2.5928798185941044), (1.0928798185941044, 3.0928798185941044), (1.5928798185941044, 3.5928798185941044), (2.0928798185941044, 4.092879818594104)]
first chunk: 0.09287981859410431 2.0928798185941044 [(0.09287981859410431, 2.0928798185941044, array([ 0.00054932,  0.00057983,  0.00064087, ..., -0.25924683,
       -0.27264404, -0.2789917 ], shape=(88200,), dtype=float32)), (0.5928798185941043, 2.5928798185941044, array([0.04928589, 0.05175781, 0.05276489, ..., 0.14593506, 0.09613037,
       0.07598877], shape=(88200,), dtype=float32)), (1.0928798185941044, 3.0928798185941044, array([-0.02212524, -0.02264404, -0.02313232, ...,  0.05499268,
        0.05331421,  0.05163574], shape=(88200,), dtype=float32)), (1.5928798185941044, 3.5928798185941044, array([0.03231812, 0.16351318, 0.01766968, ..., 0.04180908, 0.03808594,
       0.04138184], shape=(88200,), dtype=float32)), (

<h1> Tita-Net

In [12]:
import nemo.collections.asr as nemo_asr

NEMO_PATH = "../../data/models/titanet_large/speakerverification_en_titanet_large.nemo"
model = nemo_asr.models.EncDecSpeakerLabelModel.restore_from(NEMO_PATH)


[NeMo W 2026-02-23 18:19:33 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-02-23 18:19:33 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2026-02-23 18:19:33 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-23 18:19:33 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /home/aidarka/projects/conf-bot/vad-conf-bot/data/models/titanet_large/speakerverification_en_titanet_large.nemo.


In [15]:
import numpy as np
import torch
from typing import List, Tuple

def compute_embeddings_nemo(
    model,
    audio_int16: np.ndarray,
    sr: int,
    windows: List[Tuple[float, float]],
    slice_waveform_int16,
    window_sec: float = 2.0,
    batch_size: int = 128,
    l2_normalize: bool = True,
):
    """
    Returns:
      embeddings: np.ndarray shape (N, D)
      meta: list of (t0, t1) for each embedding row
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()

    T = int(round(window_sec * sr))  # samples per window

    def iter_batches(items, bs):
        for i in range(0, len(items), bs):
            yield items[i:i+bs]

    embs = []
    meta = []

    with torch.no_grad():
        for batch_windows in iter_batches(windows, batch_size):
            B = len(batch_windows)
            batch = np.zeros((B, T), dtype=np.float32)

            for j, (t0, t1) in enumerate(batch_windows):
                x = slice_waveform_int16(audio_int16, sr, t0, t1).astype(np.float32) / 32768.0

                # на всякий случай (должно быть ровно T, если окна корректно сформированы)
                if len(x) >= T:
                    batch[j] = x[:T]
                else:
                    batch[j, :len(x)] = x

            sig = torch.from_numpy(batch).to(device)
            sig_len = torch.full((B,), T, device=device, dtype=torch.long)

            logits, emb = model(input_signal=sig, input_signal_length=sig_len)
            emb = emb.detach().cpu().numpy()  # (B, 192)

            embs.append(emb)
            meta.extend(batch_windows)

    embeddings = np.concatenate(embs, axis=0)  # (N, 192)

    if l2_normalize:
        embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)

    return embeddings, meta

In [ ]:
embeddings, meta = compute_embeddings_nemo(
    model=model,
    audio_int16=audio,
    sr=sr,
    windows=windows,
    slice_waveform_int16=slice_waveform_int16,
    window_sec=2.0,
    batch_size=128,
)

print("embeddings:", embeddings.shape)  # (N, 192)
print("first meta:", meta[0])

embeddings: (122, 192)
first meta: (0.09287981859410431, 2.0928798185941044)


In [17]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-12))

print("cos(0,1):", cosine(embeddings[0], embeddings[1]))
print("cos(0,-1):", cosine(embeddings[0], embeddings[-1]))

cos(0,1): 0.9337085485458374
cos(0,-1): 0.5862801671028137
